In [1]:
import pandas as pd
import numpy as np
import requests
from pathlib import Path
from bs4 import BeautifulSoup
import unicodedata

# ── Change this one line if your CSVs live elsewhere ──────────────────────────
DATA_DIR = Path.home() / "Downloads"

# ── Fetch Wikipedia squad page ONCE; reuse html everywhere ───────────────────
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/124.0.0.0 Safari/537.36"
}
url  = "https://en.wikipedia.org/wiki/2026_FIFA_World_Cup_squads"
html = requests.get(url, headers=headers).text
tables = pd.read_html(html)
print(f"Total tables found: {len(tables)}")

C:\Users\makis\AppData\Local\Temp\ipykernel_4980\1293630688.py:19: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


Total tables found: 55


In [2]:
squad_tables = []
for t in tables:
    cols = [str(c).lower() for c in t.columns]
    if any(kw in cols for kw in ["player", "pos.", "date of birth (age)"]):
        squad_tables.append(t)
print(f"Squad tables: {len(squad_tables)}")

Squad tables: 48


In [3]:
soup = BeautifulSoup(html, "html.parser")   # reuses html from Cell 0

team_names = []
for header in soup.find_all(["h2", "h3"]):
    text = header.get_text(strip=True).replace("[edit]", "")
    if "Group" in text or "Squads" in text:
        continue
    if header.find_next("table", {"class": "wikitable"}):
        team_names.append(text)

print(team_names)

['Contents', 'Czech Republic', 'Mexico', 'South Africa', 'South Korea', 'Bosnia and Herzegovina', 'Canada', 'Qatar', 'Switzerland', 'Brazil', 'Haiti', 'Morocco', 'Scotland', 'Australia', 'Paraguay', 'Turkey', 'United States', 'Curaçao', 'Ecuador', 'Germany', 'Ivory Coast', 'Japan', 'Netherlands', 'Sweden', 'Tunisia', 'Belgium', 'Egypt', 'Iran', 'New Zealand', 'Cape Verde', 'Saudi Arabia', 'Spain', 'Uruguay', 'France', 'Iraq', 'Norway', 'Senegal', 'Algeria', 'Argentina', 'Austria', 'Jordan', 'Colombia', 'DR Congo', 'Portugal', 'Uzbekistan', 'Croatia', 'England', 'Ghana', 'Panama', 'Statistics', 'Age', 'Player representation by club', 'Player representation by league system', 'Player representation by club confederation', 'Average age of squads', 'Coach representation by country']


In [4]:
teams_to_drop = [
    "Contents", "Statistics", "Age",
    "Player representation by club",
    "Player representation by league system",
    "Player representation by club confederation",
    "Average age of squads",
    "Coach representation by country",
]
team_names = [n for n in team_names if n not in teams_to_drop]
print(team_names)
print(f"{len(team_names)} teams, {len(squad_tables)} squad tables")


assert len(team_names) == len(squad_tables), \
    f"Mismatch: {len(team_names)} names vs {len(squad_tables)} tables. Check scraping."

clean_squads = []
for team, table in zip(team_names, squad_tables):
    df = table.copy()
    df.columns = df.columns.str.lower()
    df["team"] = team
    clean_squads.append(df)

squads_df = pd.concat(clean_squads, ignore_index=True)
print(squads_df.shape)
squads_df.head()

['Czech Republic', 'Mexico', 'South Africa', 'South Korea', 'Bosnia and Herzegovina', 'Canada', 'Qatar', 'Switzerland', 'Brazil', 'Haiti', 'Morocco', 'Scotland', 'Australia', 'Paraguay', 'Turkey', 'United States', 'Curaçao', 'Ecuador', 'Germany', 'Ivory Coast', 'Japan', 'Netherlands', 'Sweden', 'Tunisia', 'Belgium', 'Egypt', 'Iran', 'New Zealand', 'Cape Verde', 'Saudi Arabia', 'Spain', 'Uruguay', 'France', 'Iraq', 'Norway', 'Senegal', 'Algeria', 'Argentina', 'Austria', 'Jordan', 'Colombia', 'DR Congo', 'Portugal', 'Uzbekistan', 'Croatia', 'England', 'Ghana', 'Panama']
48 teams, 48 squad tables
(1244, 8)


,no.,pos.,player,date of birth (age),caps,goals,club,team
0,1,GK,Matěj Kovář,"May 17, 2000 (aged 26)",20,0,PSV Eindhoven,Czech Republic
1,2,DF,David Zima,"November 8, 2000 (aged 25)",25,1,Slavia Prague,Czech Republic
2,3,DF,Tomáš Holeš,"March 31, 1993 (aged 33)",41,2,Slavia Prague,Czech Republic
3,4,DF,Robin Hranáč,"January 29, 2000 (aged 26)",14,1,TSG Hoffenheim,Czech Republic
4,5,DF,Vladimír Coufal,"August 22, 1992 (aged 33)",62,2,TSG Hoffenheim,Czech Republic


In [5]:
%%html
<style>
#notebook-container { width: 98% !important; }
div.output_scroll { height: unset !important; }
div.output_area pre { white-space: pre; overflow-x: auto; }
table.dataframe { display: block; overflow-x: auto; white-space: nowrap; }
</style>

In [6]:
players_df  = pd.read_csv(DATA_DIR / "players.csv")
valuations  = pd.read_csv(DATA_DIR / "player_valuations.csv")

valuations["date"] = pd.to_datetime(valuations["date"])

# Latest market value per player
latest_vals = (
    valuations
    .sort_values("date")
    .groupby("player_id")
    .tail(1)
    .reset_index(drop=True)
)



players_df_clean = players_df.drop(columns=["market_value_in_eur"], errors="ignore")
players_latest = players_df_clean.merge(
    latest_vals[["player_id", "market_value_in_eur"]],
    on="player_id",
    how="left"
)

print(f"Missing market values: {players_df['market_value_in_eur'].isna().sum()}")
print(players_latest.columns.tolist())

Missing market values: 16194
['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'highest_market_value_in_eur', 'market_value_in_eur']


In [7]:
def normalize(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = s.encode("ascii", "ignore").decode("utf-8")
    return s.lower().strip()

squads_df["player_clean"]     = squads_df["player"].apply(normalize)
players_latest["player_clean"] = players_latest["name"].apply(normalize)

players_deduped = (
    players_latest
    .sort_values("market_value_in_eur", ascending=False)   # FIX: no _y suffix
    .drop_duplicates(subset="player_clean", keep="first")
)

merged = squads_df.merge(
    players_deduped[["player_clean", "market_value_in_eur"]],   # FIX: no _y
    on="player_clean",
    how="left"
)
print(merged.shape)
merged.head()

(1244, 10)


,no.,pos.,player,date of birth (age),caps,goals,club,team,player_clean,market_value_in_eur
0,1,GK,Matěj Kovář,"May 17, 2000 (aged 26)",20,0,PSV Eindhoven,Czech Republic,matej kovar,7000000.0
1,2,DF,David Zima,"November 8, 2000 (aged 25)",25,1,Slavia Prague,Czech Republic,david zima,4000000.0
2,3,DF,Tomáš Holeš,"March 31, 1993 (aged 33)",41,2,Slavia Prague,Czech Republic,tomas holes,NaN
3,4,DF,Robin Hranáč,"January 29, 2000 (aged 26)",14,1,TSG Hoffenheim,Czech Republic,robin hranac,7000000.0
4,5,DF,Vladimír Coufal,"August 22, 1992 (aged 33)",62,2,TSG Hoffenheim,Czech Republic,vladimir coufal,2700000.0


In [8]:
print("Unique players:", merged["player_clean"].nunique())
merged[merged.duplicated(subset="player_clean", keep=False)][["player_clean", "team"]].head(20)

Unique players: 1243


,player_clean,team
819,emiliano martinez,Uruguay
982,emiliano martinez,Argentina


In [9]:
results = pd.read_csv(DATA_DIR / "results.csv")
results["date"] = pd.to_datetime(results["date"])

wc_teams = squads_df["team"].unique()

results_filtered = results[
    (results["date"] >= "2010-01-01") &
    (results["home_team"].isin(wc_teams) | results["away_team"].isin(wc_teams))
]

keep_tournaments = [
    "FIFA World Cup", "FIFA World Cup qualification",
    "UEFA Euro qualification", "UEFA Euro",
    "African Cup of Nations qualification", "African Cup of Nations",
    "Gold Cup", "Copa América", "Copa América qualification",
    "AFC Asian Cup", "AFC Asian Cup qualification",
    "UEFA Nations League", "CONCACAF Nations League",
    "Confederations Cup",
]

results_competitive = results_filtered[
    results_filtered["tournament"].isin(keep_tournaments)
].copy()

print(results_competitive.shape)


wc_teams_set  = set(wc_teams)
results_teams = set(results_competitive["home_team"]) | set(results_competitive["away_team"])
missing_teams = wc_teams_set - results_teams
if missing_teams:
    print("WARNING — teams with no competitive results (will get neutral stats):", missing_teams)

(4418, 9)


In [10]:
team_baseline_temp = {
    # North/Central America & Caribbean
    "Mexico": 24.5, "United States": 22.0, "Canada": 18.0,
    "Panama": 28.0, "Haiti": 29.0, "Curaçao": 29.0,
    # South America
    "Brazil": 27.0, "Argentina": 20.0, "Colombia": 24.0,
    "Uruguay": 18.0, "Paraguay": 26.0, "Ecuador": 22.0,
    # Europe
    "France": 15.0, "England": 13.0, "Spain": 20.0,
    "Portugal": 18.0, "Germany": 13.0, "Netherlands": 12.0,
    "Belgium": 12.0, "Croatia": 17.0, "Switzerland": 13.0,
    "Austria": 13.0, "Sweden": 10.0, "Norway": 9.0,
    "Denmark": 11.0, "Scotland": 11.0, "Poland": 13.0,
    "Czech Republic": 13.0, "Bosnia and Herzegovina": 15.0,
    "Turkey": 20.0, "Albania": 18.0, "Ukraine": 14.0,
    # Africa
    "Morocco": 25.0, "Senegal": 29.0, "Ivory Coast": 29.0,
    "Ghana": 29.0, "Egypt": 30.0, "Tunisia": 25.0,
    "Algeria": 25.0, "DR Congo": 28.0, "Cape Verde": 27.0,
    "South Africa": 22.0,
    # Asia/Middle East
    "Japan": 22.0, "South Korea": 20.0, "Iran": 25.0,
    "Saudi Arabia": 35.0, "Qatar": 35.0, "Iraq": 35.0,
    "Jordan": 30.0, "Uzbekistan": 25.0, "Australia": 22.0,
    "New Zealand": 15.0,
}

pressing_intensity = {
    
    "Austria": 0.95, "Canada": 0.90, "Germany": 0.88, "Spain": 0.86,
    "Japan": 0.85, "Sweden": 0.82, "Curaçao": 0.82, "United States": 0.80,
    "Portugal": 0.78, "Netherlands": 0.75, "England": 0.74, "Belgium": 0.72,
    "Norway": 0.65, "Uruguay": 0.70, "South Korea": 0.70,
    "Morocco": 0.68, "Senegal": 0.68, "Colombia": 0.67,
    "France": 0.72, "Argentina": 0.60, "Brazil": 0.58, "Mexico": 0.55,
    "Croatia": 0.50, "Switzerland": 0.50, "Ecuador": 0.50,
    "Australia": 0.48, "Ivory Coast": 0.48, "Turkey": 0.48,
    "Scotland": 0.45, "Czech Republic": 0.45,
    "Bosnia and Herzegovina": 0.42, "Algeria": 0.40,
    "Paraguay": 0.38, "Haiti": 0.38, "Panama": 0.35, "South Africa": 0.35,
    "Egypt": 0.32, "Ghana": 0.30, "Uzbekistan": 0.30, "Jordan": 0.25,
    "Iraq": 0.20, "Tunisia": 0.20,
    "DR Congo": 0.18, "Cape Verde": 0.18,
    "Iran": 0.15, "New Zealand": 0.15,
    "Saudi Arabia": 0.10, "Qatar": 0.10,
}

venue_heat = {
    "San Francisco": 18.0, "Seattle": 22.0, "Vancouver": 22.0,
    "Mexico City": 24.5, "Guadalajara": 27.5, "Toronto": 28.5,
    "Boston": 28.5, "Philadelphia": 30.5, "New York": 30.5,
    "Los Angeles": 27.0, "Kansas City": 32.5, "Atlanta": 35.0,
    "Miami": 36.5, "Houston": 39.5, "Dallas": 38.5, "Monterrey": 38.5,
}

def heat_stress(team, venue):
    return max(0, venue_heat[venue] - team_baseline_temp[team])

def heat_diff(team_a, team_b, venue):
    return heat_stress(team_a, venue) - heat_stress(team_b, venue)

def apply_heat_adjustment(p, team_a, team_b, venue, alpha_heat=0.002):
    p_adj = p - alpha_heat * heat_diff(team_a, team_b, venue)
    return max(0.01, min(0.99, p_adj))

def apply_pressing_adjustment(p, team_a, team_b, venue, alpha_press=0.003):
    press_diff    = pressing_intensity[team_a] - pressing_intensity[team_b]
    heat_severity = venue_heat[venue] / 40.0
    p_adj = p - alpha_press * press_diff * heat_severity
    return max(0.01, min(0.99, p_adj))

In [11]:
def compute_team_stats(team, df, since="2018-01-01"):
    matches = df[
        ((df["home_team"] == team) | (df["away_team"] == team)) &
        (df["date"] >= since)
    ]

    
    if len(matches) == 0:
        return {
            "team": team, "matches": 0,
            "win_rate": 0.33, "draw_rate": 0.33, "loss_rate": 0.33,
            "avg_goals_scored": 1.0, "avg_goals_conceded": 1.0,
            "goal_difference": 0.0,
        }

    rows = []
    for _, row in matches.iterrows():
        if row["home_team"] == team:
            gf, ga = row["home_score"], row["away_score"]
        else:
            gf, ga = row["away_score"], row["home_score"]
        result = "W" if gf > ga else ("D" if gf == ga else "L")
        rows.append({"gf": gf, "ga": ga, "result": result})

    rdf = pd.DataFrame(rows)
    n   = len(rdf)
    return {
        "team":               team,
        "matches":            n,
        "win_rate":           (rdf["result"] == "W").sum() / n,
        "draw_rate":          (rdf["result"] == "D").sum() / n,
        "loss_rate":          (rdf["result"] == "L").sum() / n,
        "avg_goals_scored":   rdf["gf"].mean(),
        "avg_goals_conceded": rdf["ga"].mean(),
        "goal_difference":    (rdf["gf"] - rdf["ga"]).mean(),
    }

team_stats = pd.DataFrame([compute_team_stats(t, results_competitive) for t in wc_teams])
team_stats = team_stats.sort_values("win_rate", ascending=False)
print(team_stats.head(20))

             team  matches  win_rate  draw_rate  loss_rate  avg_goals_scored  \
27    New Zealand       14  0.714286   0.000000   0.285714          4.272727   
10        Morocco       70  0.671429   0.185714   0.142857          2.029851   
20          Japan       59  0.661017   0.118644   0.220339          2.625000   
35        Senegal       76  0.657895   0.210526   0.131579          1.726027   
26           Iran       55  0.636364   0.145455   0.218182          2.192308   
24        Belgium       82  0.634146   0.134146   0.231707          2.443038   
32         France       87  0.632184   0.218391   0.149425          2.083333   
36        Algeria       62  0.629032   0.209677   0.161290          2.169492   
19    Ivory Coast       63  0.619048   0.190476   0.190476          1.800000   
30          Spain       86  0.616279   0.255814   0.127907          2.397590   
15  United States       67  0.611940   0.164179   0.223881          2.109375   
45        England       87  0.609195   0

In [12]:
games       = pd.read_csv(DATA_DIR / "games.csv")
appearances = pd.read_csv(DATA_DIR / "appearances.csv")
clubs       = pd.read_csv(DATA_DIR / "clubs.csv")
valuations  = pd.read_csv(DATA_DIR / "player_valuations.csv")

valuations["date"] = pd.to_datetime(valuations["date"])
games["date"]      = pd.to_datetime(games["date"])

print("games:",       games.shape)
print("appearances:", appearances.shape)
print("clubs:",       clubs.shape)
print("valuations:",  valuations.shape)
print("\ngames columns:",       games.columns.tolist())
print("appearances columns:", appearances.columns.tolist())

games: (88807, 23)
appearances: (1885688, 13)
clubs: (796, 17)
valuations: (507815, 6)

games columns: ['game_id', 'competition_id', 'season', 'round', 'date', 'home_club_id', 'away_club_id', 'home_club_goals', 'away_club_goals', 'home_club_position', 'away_club_position', 'home_club_manager_name', 'away_club_manager_name', 'stadium', 'attendance', 'referee', 'url', 'home_club_formation', 'away_club_formation', 'home_club_name', 'away_club_name', 'aggregate', 'competition_type']
appearances columns: ['appearance_id', 'game_id', 'player_id', 'player_club_id', 'player_current_club_id', 'date', 'player_name', 'competition_id', 'yellow_cards', 'red_cards', 'goals', 'assists', 'minutes_played']


In [13]:
print(games["competition_type"].value_counts())

competition_type
domestic_league              63566
domestic_cup                 13514
other                         7940
international_cup             3117
national_team_competition      670
Name: count, dtype: int64


In [14]:
# ── Prime score function (Branquinho et al. Gaussian) ────────────────────────
def calculate_prime_score(age):
    if pd.isna(age) or age <= 0:
        return 0.7          # Conservative baseline for missing/invalid ages
    PEAK_AGE = 25.5         # Midpoint of endurance (24.8) & explosive (26.0) peaks
    SIGMA    = 3.5          # Gradual rise, sharp drop after 32
    return np.exp(-((age - PEAK_AGE) ** 2) / (2 * SIGMA ** 2))


league_games = games[
    (games["competition_type"] == "domestic_league") &
    (games["date"] >= "2005-01-01")
].copy()
print("league_games:", league_games.shape)


league_app = appearances[appearances["game_id"].isin(league_games["game_id"])].copy()
league_app["date"] = pd.to_datetime(league_app["date"])
print("league_app:", league_app.shape)


league_app_vals = league_app.merge(
    valuations[["player_id", "date", "market_value_in_eur"]],
    on="player_id",
    how="left",
    suffixes=("", "_val")
)
league_app_vals = league_app_vals[league_app_vals["date_val"] <= league_app_vals["date"]]
league_app_vals = (
    league_app_vals
    .sort_values("date_val")
    .groupby("appearance_id")
    .tail(1)
)
print("After value join:", league_app_vals.shape)
print(league_app_vals["market_value_in_eur"].isna().sum(), "missing market values")


players_dob = players_df[["player_id", "date_of_birth"]].copy()
players_dob["date_of_birth"] = pd.to_datetime(players_dob["date_of_birth"])
league_app_vals = league_app_vals.merge(players_dob, on="player_id", how="left")
league_app_vals["match_age"] = (
    league_app_vals["date"] - league_app_vals["date_of_birth"]
).dt.days / 365.25
league_app_vals["prime_score"] = league_app_vals["match_age"].apply(calculate_prime_score)
print("Age + prime score attached.")

league_games: (63566, 23)
league_app: (1603375, 13)
After value join: (1573560, 15)
0 missing market values
Age + prime score attached.


In [15]:
def gini(values):
    values = sorted(values)
    n = len(values)
    if n == 0 or sum(values) == 0:
        return 0
    cumsum = sum((2 * (i + 1) - n - 1) * v for i, v in enumerate(values))
    return cumsum / (n * sum(values))

squad_values = (
    league_app_vals
    .groupby(["game_id", "player_club_id"])
    .agg(
        total_value    =("market_value_in_eur", "sum"),
        avg_value      =("market_value_in_eur", "mean"),
        num_players    =("market_value_in_eur", "count"),
        avg_prime_score=("prime_score",          "mean"),
    )
    .reset_index()
)

gini_vals = (
    league_app_vals
    .groupby(["game_id", "player_club_id"])["market_value_in_eur"]
    .apply(gini)
    .reset_index()
    .rename(columns={"market_value_in_eur": "gini"})
)

squad_values = squad_values.merge(gini_vals, on=["game_id", "player_club_id"])
print(squad_values.shape)
squad_values.head()

(111985, 7)


,game_id,player_club_id,total_value,avg_value,num_players,avg_prime_score,gini
0,2222535,16,166000000.0,1.185714e+07,14,0.647269,0.398451
1,2222535,86,57300000.0,4.407692e+06,13,0.756922,0.336153
2,2222536,33,88500000.0,7.375000e+06,12,0.604701,0.368644
3,2222536,42,47650000.0,3.403571e+06,14,0.643615,0.365987
4,2222537,18,68150000.0,4.867857e+06,14,0.406867,0.402526


In [16]:
records = []
for _, row in league_games.iterrows():
    records.append({
        "game_id": row["game_id"], "date": row["date"],
        "club_id": row["home_club_id"],
        "win": 1 if row["home_club_goals"] > row["away_club_goals"] else 0,
    })
    records.append({
        "game_id": row["game_id"], "date": row["date"],
        "club_id": row["away_club_id"],
        "win": 1 if row["away_club_goals"] > row["home_club_goals"] else 0,
    })

form_records = pd.DataFrame(records).sort_values(["club_id", "date"])
form_records["form"] = (
    form_records.groupby("club_id")["win"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
form_records["form"] = form_records["form"].fillna(0.5)
print(form_records.shape)
form_records.head(10)

(127132, 5)


,game_id,date,club_id,win,form
17002,2460633,2014-08-23,3,0,0.500000
17021,2460642,2014-08-30,3,1,0.000000
17041,2460652,2014-09-13,3,0,0.500000
17056,2460660,2014-09-21,3,0,0.333333
17071,2460667,2014-09-24,3,0,0.250000
17092,2460678,2014-09-27,3,0,0.200000
17113,2460688,2014-10-04,3,0,0.200000
17128,2460696,2014-10-18,3,1,0.000000
17147,2460705,2014-10-24,3,1,0.200000
17164,2460714,2014-11-02,3,0,0.400000


In [17]:
home_form = form_records[["game_id", "club_id", "form"]].rename(
    columns={"club_id": "home_club_id", "form": "home_form"})
away_form = form_records[["game_id", "club_id", "form"]].rename(
    columns={"club_id": "away_club_id", "form": "away_form"})

# Build home and away AFTER squad_values has avg_prime_score
home = games[["game_id", "home_club_id", "home_club_goals"]].merge(
    squad_values.rename(columns={
        "player_club_id":  "home_club_id",
        "total_value":     "home_total_value",
        "avg_value":       "home_avg_value",
        "gini":            "home_gini",
        "num_players":     "home_num_players",
        "avg_prime_score": "home_avg_prime_score",
    }),
    on=["game_id", "home_club_id"]
)

away = games[["game_id", "away_club_id", "away_club_goals"]].merge(
    squad_values.rename(columns={
        "player_club_id":  "away_club_id",
        "total_value":     "away_total_value",
        "avg_value":       "away_avg_value",
        "gini":            "away_gini",
        "num_players":     "away_num_players",
        "avg_prime_score": "away_avg_prime_score",  
    }),
    on=["game_id", "away_club_id"]
)



train_df = home.merge(
    away[["game_id", "away_club_id", "away_club_goals", "away_total_value", 
          "away_avg_value", "away_gini", "away_num_players", "away_avg_prime_score"]],
    on="game_id"
)

train_df = train_df.merge(home_form, on=["game_id", "home_club_id"], how="left")
train_df = train_df.merge(away_form, on=["game_id", "away_club_id"], how="left")

train_df["home_form"] = train_df["home_form"].fillna(0.5)
train_df["away_form"] = train_df["away_form"].fillna(0.5)

train_df["result"] = train_df.apply(
    lambda r: 1 if r["home_club_goals"] > r["away_club_goals"]
    else (0 if r["home_club_goals"] == r["away_club_goals"] else -1), axis=1
)

train_df["total_value_diff"] = train_df["home_total_value"]     - train_df["away_total_value"]
train_df["avg_value_diff"]   = train_df["home_avg_value"]       - train_df["away_avg_value"]
train_df["gini_diff"]        = train_df["home_gini"]            - train_df["away_gini"]
train_df["form_diff"]        = train_df["home_form"]            - train_df["away_form"]
train_df["prime_diff"]       = train_df["home_avg_prime_score"] - train_df["away_avg_prime_score"]

print(train_df.shape)
print(train_df["result"].value_counts())

(55935, 23)
result
 1    24842
-1    17046
 0    14047
Name: count, dtype: int64


In [18]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

features = ["total_value_diff", "avg_value_diff", "gini_diff", "form_diff", "prime_diff"]

X = train_df[features]
y = train_df["result"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

importances = pd.DataFrame({
    "feature":    features,
    "importance": model.feature_importances_,
}).sort_values("importance", ascending=False)
print(importances)

Accuracy: 0.49521766335925627
              precision    recall  f1-score   support

          -1       0.46      0.49      0.47      3319
           0       0.28      0.13      0.18      2809
           1       0.56      0.70      0.62      5059

    accuracy                           0.50     11187
   macro avg       0.43      0.44      0.43     11187
weighted avg       0.46      0.50      0.47     11187

            feature  importance
1    avg_value_diff    0.249339
0  total_value_diff    0.241046
4        prime_diff    0.222596
2         gini_diff    0.221729
3         form_diff    0.065288


In [19]:

merged["market_value_in_eur"] = merged["market_value_in_eur"].fillna(0)


# Extract age from Wikipedia's 'date of birth (age)' column safely
dob_col = next((c for c in merged.columns if "date of birth" in c.lower() or "age" in c.lower()), None)

if dob_col:
    # Captures digits inside parentheses even if preceded by "aged " or "age "
    extracted_age = merged[dob_col].astype(str).str.extract(r"\((?:aged\s+|age\s+)?(\d+)\)")[0]
    
    # Cast to float, and fill missing values with a realistic default squad mean (26)
    merged["age"] = pd.to_numeric(extracted_age, errors='coerce').fillna(26.0)
elif "age" not in merged.columns:
    merged["age"] = 26.0  # Realistic baseline fallback for international players

merged["prime_score"] = merged["age"].apply(calculate_prime_score)

team_features = (
    merged.groupby("team")
    .agg(
        total_value    =("market_value_in_eur", "sum"),
        avg_value      =("market_value_in_eur", "mean"),
        num_players    =("market_value_in_eur", "count"),
        avg_prime_score=("prime_score",          "mean"),
        gini           =("market_value_in_eur", gini),
    )
    .reset_index()
)
print(team_features.sort_values("total_value", ascending=False).head(10))

# Check for any WC teams missing from team_features
groups_check = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}
all_teams   = [t for teams in groups_check.values() for t in teams]
known_teams = set(team_features["team"])
missing     = [t for t in all_teams if t not in known_teams]
print("Missing from team_features:", missing)

           team   total_value     avg_value  num_players  avg_prime_score  \
17       France  1.283000e+09  4.934615e+07           26         0.662843   
16      England  1.229000e+09  4.726923e+07           26         0.638766   
33     Portugal  1.007500e+09  3.875000e+07           26         0.655513   
40        Spain  8.910000e+08  3.426923e+07           26         0.616682   
18      Germany  8.855000e+08  3.405769e+07           26         0.590267   
28  Netherlands  8.171000e+08  3.142692e+07           26         0.670373   
1     Argentina  7.465000e+08  2.986000e+07           25         0.566759   
6        Brazil  7.062000e+08  2.716154e+07           26         0.482865   
30       Norway  4.437000e+08  1.706538e+07           26         0.762497   
41       Sweden  4.248000e+08  1.633846e+07           26         0.708967   

        gini  
17  0.364260  
16  0.412937  
33  0.400744  
40  0.391393  
18  0.470117  
28  0.412326  
1   0.552552  
6   0.615221  
30  0.738224  
41

In [20]:
def compute_international_form(team, df, n=20):
    matches = df[(df["home_team"] == team) | (df["away_team"] == team)]
    matches = matches.sort_values("date").tail(n)
    if len(matches) == 0:
        return 0.5
    form_score = sum(
        1 for _, row in matches.iterrows()
        if (row["home_team"] == team and row["home_score"] > row["away_score"]) or
           (row["away_team"] == team and row["away_score"] > row["home_score"])
    )
    return form_score / len(matches)


int_form = {
    team: compute_international_form(team, results_competitive, n=20)
    for team in wc_teams
}

team_features["int_form"] = team_features["team"].map(int_form).fillna(0.5)
print(team_features[["team", "int_form"]].sort_values("int_form", ascending=False).head(15))

           team  int_form
27      Morocco      0.75
0       Algeria      0.70
16      England      0.70
37      Senegal      0.70
30       Norway      0.65
40        Spain      0.60
23  Ivory Coast      0.60
24        Japan      0.60
29  New Zealand      0.55
13     DR Congo      0.55
44       Turkey      0.55
15        Egypt      0.55
17       France      0.55
26       Mexico      0.55
1     Argentina      0.55


In [21]:
odds_fanduel = {
    "Spain": 420, "France": 460, "England": 650, "Brazil": 850,
    "Portugal": 1000, "Argentina": 1000, "Germany": 1300, "Netherlands": 1600,
    "Belgium": 2200, "Norway": 3500, "Colombia": 4000, "Japan": 4500,
    "Morocco": 6000, "United States": 6000, "Uruguay": 6000, "Mexico": 6500,
    "Switzerland": 6500, "Croatia": 7000, "Turkey": 8000, "Ecuador": 10000,
    "Senegal": 12500, "Austria": 12500, "Canada": 17500, "Sweden": 17500,
    "Ivory Coast": 17500, "Paraguay": 20000, "Egypt": 25000, "Scotland": 30000,
    "Bosnia and Herzegovina": 40000, "Ghana": 60000, "Czech Republic": 60000,
    "South Korea": 70000, "Iran": 100000, "Tunisia": 200000, "Cape Verde": 250000,
    "Uzbekistan": 250000, "Haiti": 250000, "Panama": 250000, "Curaçao": 250000,
    "Qatar": 250000, "Saudi Arabia": 250000, "New Zealand": 250000,
    "Australia": 250000, "DR Congo": 250000, "Iraq": 250000, "Jordan": 250000,
    "South Africa": 250000, "Algeria": 250000,
}

def american_to_prob(o):
    return 100 / (o + 100) if o > 0 else abs(o) / (abs(o) + 100)

raw_probs = {t: american_to_prob(o) for t, o in odds_fanduel.items()}
total     = sum(raw_probs.values())
odds_prob = {t: p / total for t, p in raw_probs.items()}


def predict_match(team_a, team_b, venue, odds_weight=0.6):
    a = team_features[team_features["team"] == team_a].iloc[0]
    b = team_features[team_features["team"] == team_b].iloc[0]

    X = pd.DataFrame([{
        "total_value_diff": a["total_value"]     - b["total_value"],
        "avg_value_diff":   a["avg_value"]       - b["avg_value"],
        "gini_diff":        a["gini"]            - b["gini"],
        "form_diff":        a["int_form"]        - b["int_form"],
        "prime_diff":       a["avg_prime_score"] - b["avg_prime_score"],
    }])

    proba       = model.predict_proba(X)[0]
    classes     = model.classes_
    model_probs = {int(c): p for c, p in zip(classes, proba)}

    prob_a     = odds_prob.get(team_a, 0.01)
    prob_b     = odds_prob.get(team_b, 0.01)
    odds_win_a = prob_a / (prob_a + prob_b)
    odds_win_b = prob_b / (prob_a + prob_b)

    p_win_a = apply_heat_adjustment(model_probs[1],  team_a, team_b, venue)
    p_win_b = apply_heat_adjustment(model_probs[-1], team_b, team_a, venue)
    p_draw  = model_probs[0]

    p_win_a = apply_pressing_adjustment(p_win_a, team_a, team_b, venue)
    p_win_b = apply_pressing_adjustment(p_win_b, team_b, team_a, venue)

    s       = p_win_a + p_win_b + p_draw
    p_win_a /= s
    p_win_b /= s
    p_draw  /= s

    # FIX: draw now gets an odds component too so it isn't systematically suppressed
    blended_win_a = (1 - odds_weight) * p_win_a + odds_weight * odds_win_a
    blended_win_b = (1 - odds_weight) * p_win_b + odds_weight * odds_win_b
    blended_draw  = (1 - odds_weight) * p_draw  + odds_weight * model_probs[0]

    total = blended_win_a + blended_win_b + blended_draw
    return {
         1: blended_win_a / total,
         0: blended_draw  / total,
        -1: blended_win_b / total,
    }

In [22]:
from itertools import combinations
from collections import Counter

official_match_venues = {
    ("Mexico", "South Africa"): "Mexico City",
    ("Korea Republic", "Czechia"): "Guadalajara",
    ("Canada", "Bosnia and Herzegovina"): "Toronto",
    ("USA", "Paraguay"): "Los Angeles",
    ("Haiti", "Scotland"): "Boston",
    ("Australia", "Türkiye"): "Vancouver",
    ("Brazil", "Morocco"): "New York",
    ("Qatar", "Switzerland"): "San Francisco",
    ("Côte d\'Ivoire", "Ecuador"): "Philadelphia",
    ("Germany", "Curaçao"): "Houston",
    ("Netherlands", "Japan"): "Dallas",
    ("Sweden", "Tunisia"): "Monterrey",
    ("Saudi Arabia", "Uruguay"): "Miami",
    ("Spain", "Cabo Verde"): "Atlanta",
    ("IR Iran", "New Zealand"): "Los Angeles",
    ("Belgium", "Egypt"): "Seattle",
    ("France", "Senegal"): "New York",
    ("Iraq", "Norway"): "Boston",
    ("Argentina", "Algeria"): "Kansas City",
    ("Austria", "Jordan"): "San Francisco",
    ("Ghana", "Panama"): "Toronto",
    ("England", "Croatia"): "Dallas",
    ("Portugal", "Congo DR"): "Houston",
    ("Uzbekistan", "Colombia"): "Mexico City",
    ("Czechia", "South Africa"): "Atlanta",
    ("Switzerland", "Bosnia and Herzegovina"): "Los Angeles",
    ("Canada", "Qatar"): "Vancouver",
    ("Mexico", "Korea Republic"): "Guadalajara",
    ("Brazil", "Haiti"): "Philadelphia",
    ("Scotland", "Morocco"): "Boston",
    ("Türkiye", "Paraguay"): "San Francisco",
    ("USA", "Australia"): "Seattle",
    ("Germany", "Côte d\'Ivoire"): "Toronto",
    ("Ecuador", "Curaçao"): "Kansas City",
    ("Netherlands", "Sweden"): "Houston",
    ("Tunisia", "Japan"): "Monterrey",
    ("Uruguay", "Cabo Verde"): "Miami",
    ("Spain", "Saudi Arabia"): "Atlanta",
    ("Belgium", "IR Iran"): "Los Angeles",
    ("New Zealand", "Egypt"): "Vancouver",
    ("Norway", "Senegal"): "New York",
    ("France", "Iraq"): "Philadelphia",
    ("Argentina", "Austria"): "Dallas",
    ("Jordan", "Algeria"): "San Francisco",
    ("England", "Ghana"): "Boston",
    ("Panama", "Croatia"): "Toronto",
    ("Portugal", "Uzbekistan"): "Houston",
    ("Colombia", "Congo DR"): "Guadalajara",
    ("Scotland", "Brazil"): "Miami",
    ("Morocco", "Haiti"): "Atlanta",
    ("Switzerland", "Canada"): "Vancouver",
    ("Bosnia and Herzegovina", "Qatar"): "Seattle",
    ("Czechia", "Mexico"): "Mexico City",
    ("South Africa", "Korea Republic"): "Monterrey",
    ("Curaçao", "Côte d\'Ivoire"): "Philadelphia",
    ("Ecuador", "Germany"): "New York",
    ("Japan", "Sweden"): "Dallas",
    ("Tunisia", "Netherlands"): "Kansas City",
    ("Türkiye", "USA"): "Los Angeles",
    ("Paraguay", "Australia"): "San Francisco",
    ("Norway", "France"): "Boston",
    ("Senegal", "Iraq"): "Toronto",
    ("Egypt", "IR Iran"): "Seattle",
    ("New Zealand", "Belgium"): "Vancouver",
    ("Cabo Verde", "Saudi Arabia"): "Houston",
    ("Uruguay", "Spain"): "Guadalajara",
    ("Panama", "England"): "New York",
    ("Croatia", "Ghana"): "Philadelphia",
    ("Algeria", "Austria"): "Kansas City",
    ("Jordan", "Argentina"): "Dallas",
    ("Colombia", "Portugal"): "Miami",
    ("Congo DR", "Uzbekistan"): "Atlanta",
}

knockout_venues = {
    # Round of 32
    73: "Los Angeles", 74: "Boston",      75: "Monterrey",   76: "Houston",
    77: "New York",    78: "Dallas",       79: "Mexico City", 80: "Atlanta",
    81: "San Francisco", 82: "Seattle",   83: "Toronto",     84: "Los Angeles",
    85: "Vancouver",   86: "Miami",        87: "Kansas City", 88: "Dallas",
    # Round of 16
    89: "Philadelphia", 90: "Houston",    91: "New York",    92: "Mexico City",
    93: "Dallas",       94: "Seattle",    95: "Atlanta",     96: "Vancouver",
    # Quarter-finals
    97: "Boston",       98: "Los Angeles", 99: "Miami",      100: "Kansas City",
    # Semi-finals
    101: "Dallas",      102: "Atlanta",
    # Bronze final
    103: "Miami",
    # Final
    104: "New York",
}

groups = {
    "A": ["Mexico", "South Africa", "South Korea", "Czech Republic"],
    "B": ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "C": ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D": ["United States", "Paraguay", "Australia", "Turkey"],
    "E": ["Germany", "Curaçao", "Ivory Coast", "Ecuador"],
    "F": ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G": ["Belgium", "Egypt", "Iran", "New Zealand"],
    "H": ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I": ["France", "Senegal", "Iraq", "Norway"],
    "J": ["Argentina", "Algeria", "Austria", "Jordan"],
    "K": ["Portugal", "DR Congo", "Uzbekistan", "Colombia"],
    "L": ["England", "Croatia", "Ghana", "Panama"],
}

venue_name_map = {
    "Ivory Coast":   "Côte d\'Ivoire",
    "Turkey":        "Türkiye",
    "Iran":          "IR Iran",
    "DR Congo":      "Congo DR",
    "Czech Republic":"Czechia",
    "United States": "USA",
}

def get_venue(a, b):
    a_m = venue_name_map.get(a, a)
    b_m = venue_name_map.get(b, b)
    venue = (
        official_match_venues.get((a_m, b_m)) or
        official_match_venues.get((b_m, a_m)) or
        official_match_venues.get((a, b)) or
        official_match_venues.get((b, a))
    )
    return venue if venue else "Dallas"  # fallback

# ── Build matchup cache ───────────────────────────────────────────────────────
all_wc_teams = [t for teams in groups.values() for t in teams]
all_venues   = set(knockout_venues.values())
matchup_cache = {}

for grp, teams in groups.items():
    for h, aw in combinations(teams, 2):
        venue = get_venue(h, aw)
        for a, b in [(h, aw), (aw, h)]:
            key = (str(a), str(b), venue)
            if key not in matchup_cache:
                matchup_cache[key] = predict_match(str(a), str(b), venue)

for venue in all_venues:
    for ta, tb in combinations(all_wc_teams, 2):
        for a, b in [(ta, tb), (tb, ta)]:
            key = (a, b, venue)
            if key not in matchup_cache:
                matchup_cache[key] = predict_match(a, b, venue)

print("Cache built:", len(matchup_cache), "matchups")

Cache built: 33844 matchups


In [23]:
def get_lambdas(team_a, team_b):
    a = team_features[team_features["team"] == team_a].iloc[0]
    b = team_features[team_features["team"] == team_b].iloc[0]
    val_diff = (a["total_value"] - b["total_value"]) / 1e9
    return max(0.5, 1.5 + val_diff), max(0.5, 1.5 - val_diff)


def simulate_score(lambda_a, lambda_b, outcome, max_tries=500):
    """Rejection sampling — no clamping distortion."""
    for _ in range(max_tries):
        ga = np.random.poisson(lambda_a)
        gb = np.random.poisson(lambda_b)
        if outcome == 1  and ga > gb: return ga, gb
        if outcome == -1 and gb > ga: return ga, gb
        if outcome == 0  and ga == gb: return ga, gb
    # Rare fallback
    return {1: (1, 0), -1: (0, 1), 0: (0, 0)}[outcome]


def simulate_group(teams):
    points = {t: 0 for t in teams}
    gd     = {t: 0 for t in teams}
    gf     = {t: 0 for t in teams}

    for home, away in combinations(teams, 2):
        venue   = get_venue(home, away)
        proba   = matchup_cache[(str(home), str(away), venue)]
        outcome = np.random.choice([1, 0, -1], p=[proba[1], proba[0], proba[-1]])

        lh, la          = get_lambdas(home, away)
        goals_h, goals_a = simulate_score(lh, la, outcome)

        if outcome == 1:
            points[home] += 3
        elif outcome == -1:
            points[away] += 3
        else:
            points[home] += 1
            points[away] += 1

        gf[home] += goals_h;  gf[away] += goals_a
        gd[home] += goals_h - goals_a
        gd[away] += goals_a - goals_h

    standings = sorted(teams, key=lambda t: (points[t], gd[t], gf[t]), reverse=True)
    return [(t, points[t], gd[t], gf[t]) for t in standings]


def simulate_knockout(team_a, team_b, match_number):
    venue   = knockout_venues[match_number]
    proba   = matchup_cache[(str(team_a), str(team_b), venue)]
    outcome = np.random.choice([1, 0, -1], p=[proba[1], proba[0], proba[-1]])

    la, lb           = get_lambdas(team_a, team_b)
    simulate_score(la, lb, outcome)   

    if outcome == 1:  return str(team_a)
    if outcome == -1: return str(team_b)
    return str(team_a) if np.random.random() < 0.5 else str(team_b)  # penalties


def rank_third_place(full_standings):
    third = [
        (table[2][0], table[2][1], table[2][2], table[2][3])
        for table in full_standings.values()
    ]
    return [t[0] for t in sorted(third, key=lambda x: (x[1], x[2], x[3]), reverse=True)[:8]]

In [24]:
def simulate_tournament():
    full_standings = {g: simulate_group(teams) for g, teams in groups.items()}

    winners = {g: t[0][0] for g, t in full_standings.items()}
    runners  = {g: t[1][0] for g, t in full_standings.items()}
    best8    = rank_third_place(full_standings)

    r32 = [
        simulate_knockout(runners["A"],  runners["B"],  73),
        simulate_knockout(winners["E"],  best8[0],      74),
        simulate_knockout(winners["F"],  runners["C"],  75),
        simulate_knockout(winners["C"],  runners["F"],  76),
        simulate_knockout(winners["I"],  best8[1],      77),
        simulate_knockout(runners["E"],  runners["I"],  78),
        simulate_knockout(winners["A"],  best8[2],      79),
        simulate_knockout(winners["L"],  best8[3],      80),
        simulate_knockout(winners["D"],  best8[4],      81),
        simulate_knockout(winners["G"],  best8[5],      82),
        simulate_knockout(runners["K"],  runners["L"],  83),
        simulate_knockout(winners["H"],  runners["J"],  84),
        simulate_knockout(winners["B"],  best8[6],      85),
        simulate_knockout(winners["J"],  runners["H"],  86),
        simulate_knockout(winners["K"],  best8[7],      87),
        simulate_knockout(runners["D"],  runners["G"],  88),
    ]

    r16 = [
        simulate_knockout(r32[1],  r32[4],  89),
        simulate_knockout(r32[0],  r32[2],  90),
        simulate_knockout(r32[3],  r32[5],  91),
        simulate_knockout(r32[6],  r32[7],  92),
        simulate_knockout(r32[10], r32[11], 93),
        simulate_knockout(r32[8],  r32[9],  94),
        simulate_knockout(r32[13], r32[15], 95),
        simulate_knockout(r32[12], r32[14], 96),
    ]

    sf = [
        simulate_knockout(r16[0], r16[1], 97),
        simulate_knockout(r16[4], r16[5], 98),
        simulate_knockout(r16[2], r16[3], 99),
        simulate_knockout(r16[6], r16[7], 100),
    ]

    finalists = [
        simulate_knockout(sf[0], sf[1], 101),
        simulate_knockout(sf[2], sf[3], 102),
    ]

    return simulate_knockout(finalists[0], finalists[1], 104)


#Monte carlo
n_simulations = 10_000
win_counter   = Counter()

for _ in range(n_simulations):
    win_counter[simulate_tournament()] += 1

results_sim = pd.DataFrame(win_counter.most_common(), columns=["team", "wins"])
results_sim["probability"] = results_sim["wins"] / n_simulations * 100
print(results_sim.to_string(index=False))

                  team  wins  probability
                France  1971        19.71
                 Spain  1582        15.82
               England  1452        14.52
              Portugal   960         9.60
             Argentina   822         8.22
                Brazil   803         8.03
               Germany   636         6.36
           Netherlands   499         4.99
               Belgium   261         2.61
                Norway   143         1.43
         United States    94         0.94
              Colombia    93         0.93
                 Japan    81         0.81
               Uruguay    80         0.80
                Mexico    75         0.75
           Switzerland    62         0.62
               Croatia    60         0.60
               Morocco    58         0.58
                Turkey    54         0.54
               Austria    29         0.29
               Senegal    27         0.27
               Ecuador    24         0.24
                Sweden    22      